# WTI Crude Oil — M1 & M12 Daily Return Series

Pulls a grid of CL (WTI crude) futures settlement prices from Interactive Brokers TWS,
then constructs clean daily log-return series for the front-month (M1) and 12th-month (M12) contracts.

**Requirements:**
- TWS running locally with API enabled on port 7496
- `pip install ib_insync pandas numpy`

**Roll logic:**
On roll dates, the return is computed using the *new* contract's prices on both the prior and current day — no NaN gaps.

## 1. Configuration

In [ ]:
TWS_HOST = '127.0.0.1'
TWS_PORT = 7496
CLIENT_ID = 1

# Date range for final output
START_DATE = '2025-03-01'
END_DATE = '2026-03-09'

# Contract years to pull (must cover M12 beyond END_DATE)
YEARS = [2024, 2025, 2026, 2027]

# Historical data lookback
DURATION = '2 Y'

# Pacing delay between requests (seconds)
PACING_DELAY = 3

# Output files
GRID_CSV = 'cl_contract_grid.csv'
RETURNS_CSV = 'wti_returns.csv'

## 2. Pull Contract Grid from TWS

In [ ]:
from ib_insync import *
import pandas as pd
import numpy as np
from pandas.tseries.offsets import BDay
import time

MONTH_CODES = {
    1:'F', 2:'G', 3:'H', 4:'J', 5:'K', 6:'M',
    7:'N', 8:'Q', 9:'U', 10:'V', 11:'X', 12:'Z'
}

ib = IB()
ib.connect(TWS_HOST, TWS_PORT, clientId=CLIENT_ID)
print(f'Connected: {ib.isConnected()}')

In [ ]:
# Build list of contracts — IB uses single-digit year for 202x
contracts_to_fetch = []
for year in YEARS:
    for month in range(1, 13):
        local_symbol = f'CL{MONTH_CODES[month]}{year % 10}'
        contracts_to_fetch.append((local_symbol, year, month))

print(f'{len(contracts_to_fetch)} contracts to fetch')
print(f'First: {contracts_to_fetch[0][0]}, Last: {contracts_to_fetch[-1][0]}')

In [ ]:
all_data = {}

for local_symbol, year, month in contracts_to_fetch:
    contract = Future(
        symbol='CL',
        exchange='NYMEX',
        currency='USD',
        localSymbol=local_symbol,
        includeExpired=True
    )
    try:
        bars = ib.reqHistoricalData(
            contract,
            endDateTime='',
            durationStr=DURATION,
            barSizeSetting='1 day',
            whatToShow='TRADES',
            useRTH=True,
            formatDate=1
        )
        if bars:
            df = util.df(bars)[['date', 'close']]
            df.columns = ['date', local_symbol]
            df['date'] = pd.to_datetime(df['date'])
            df = df.set_index('date')
            all_data[local_symbol] = df
            print(f'{local_symbol}: {len(df)} bars, {df.index[0].date()} to {df.index[-1].date()}')
        else:
            print(f'{local_symbol}: no data returned')
    except Exception as e:
        print(f'{local_symbol}: error - {e}')
    time.sleep(PACING_DELAY)

ib.disconnect()
print(f'\nPulled {len(all_data)} contracts with data')

In [ ]:
# Save grid
grid = pd.concat(all_data.values(), axis=1)
for col in grid.columns:
    grid[col] = pd.to_numeric(grid[col], errors='coerce')
grid.to_csv(GRID_CSV)
print(f'Grid shape: {grid.shape}')
print(f'Date range: {grid.index[0].date()} to {grid.index[-1].date()}')
print(f'Columns: {list(grid.columns)}')

## 3. Build Expiry Calendar & Assign M1 / M12

In [ ]:
CODE_TO_MONTH = {v: k for k, v in MONTH_CODES.items()}

def get_expiry(year, month):
    """
    CME WTI rule: last trading day is 3rd business day prior to
    the 25th calendar day of the month PRECEDING delivery.
    """
    preceding = pd.Timestamp(year=year, month=month, day=1) - pd.DateOffset(months=1)
    the_25th = pd.Timestamp(year=preceding.year, month=preceding.month, day=25)
    expiry = the_25th - 3 * BDay()
    return expiry

def parse_contract(col):
    """Parse column name like CLZ4 or CLZ36 into (month, year, expiry)."""
    month_code = col[2]
    year_str = col[3:]
    year = int('202' + year_str) if len(year_str) == 1 else int('20' + year_str)
    month = CODE_TO_MONTH[month_code]
    return month, year, get_expiry(year, month)

# Parse all contracts and sort by expiry
contracts = {}
for col in grid.columns:
    try:
        month, year, expiry = parse_contract(col)
        contracts[col] = {'month': month, 'year': year, 'expiry': expiry}
    except Exception as e:
        print(f'Could not parse {col}: {e}')

sorted_contracts = sorted(contracts.items(), key=lambda x: x[1]['expiry'])
print(f'Parsed {len(contracts)} contracts')
print(f'Earliest expiry: {sorted_contracts[0][0]} ({sorted_contracts[0][1]["expiry"].date()})')
print(f'Latest expiry: {sorted_contracts[-1][0]} ({sorted_contracts[-1][1]["expiry"].date()})')

In [ ]:
# For each trading day, identify M1 and M12
records = []
for date in grid.index:
    active = [(col, meta) for col, meta in sorted_contracts if meta['expiry'] >= date]
    if len(active) >= 12:
        m1_col = active[0][0]
        m12_col = active[11][0]
        records.append({
            'date': date,
            'm1_contract': m1_col,
            'm12_contract': m12_col,
            'm1_price': float(grid.loc[date, m1_col]),
            'm12_price': float(grid.loc[date, m12_col]),
        })

df = pd.DataFrame(records).set_index('date')
print(f'Assigned M1/M12 for {len(df)} trading days')

## 4. Compute Log Returns (with Roll-Date Fix)

In [ ]:
df['m1_roll'] = df['m1_contract'] != df['m1_contract'].shift(1)
df['m12_roll'] = df['m12_contract'] != df['m12_contract'].shift(1)

# On roll dates, use the NEW contract's price on both days
m1_ret = [np.nan]
m12_ret = [np.nan]

for i in range(1, len(df)):
    date_today = df.index[i]
    date_yesterday = df.index[i - 1]

    # M1
    m1_col = df.iloc[i]['m1_contract']
    m1_today = grid.loc[date_today, m1_col] if date_today in grid.index else np.nan
    m1_yest = grid.loc[date_yesterday, m1_col] if date_yesterday in grid.index else np.nan
    if pd.notna(m1_today) and pd.notna(m1_yest) and m1_yest > 0:
        m1_ret.append(np.log(m1_today / m1_yest))
    else:
        m1_ret.append(np.nan)

    # M12
    m12_col = df.iloc[i]['m12_contract']
    m12_today = grid.loc[date_today, m12_col] if date_today in grid.index else np.nan
    m12_yest = grid.loc[date_yesterday, m12_col] if date_yesterday in grid.index else np.nan
    if pd.notna(m12_today) and pd.notna(m12_yest) and m12_yest > 0:
        m12_ret.append(np.log(m12_today / m12_yest))
    else:
        m12_ret.append(np.nan)

df['m1_ret'] = m1_ret
df['m12_ret'] = m12_ret

print(f'Roll dates — M1: {df["m1_roll"].sum()}, M12: {df["m12_roll"].sum()}')

## 5. Filter Date Range & Export

In [ ]:
df = df.loc[START_DATE:END_DATE]

df['m1_price'] = df['m1_price'].astype(float)
df['m12_price'] = df['m12_price'].astype(float)
df['m1_ret'] = df['m1_ret'].astype(float)
df['m12_ret'] = df['m12_ret'].astype(float)

df.to_csv(RETURNS_CSV)

print(f'Date range: {df.index[0].date()} to {df.index[-1].date()}')
print(f'Total trading days: {len(df)}')
print(f'M1 valid returns: {df["m1_ret"].notna().sum()}')
print(f'M12 valid returns: {df["m12_ret"].notna().sum()}')
print(f'\nM1 return stats:')
print(df['m1_ret'].describe())
print(f'\nM12 return stats:')
print(df['m12_ret'].describe())

In [ ]:
# Verify roll dates have valid returns
print('Sample around first M1 roll:')
rolls = df[df['m1_roll']]
if len(rolls) > 0:
    roll_date = rolls.index[0]
    idx = df.index.get_loc(roll_date)
    print(df[['m1_contract', 'm1_price', 'm1_ret', 'm1_roll']].iloc[max(0, idx-1):idx+2])

print(f'\nLast 5 rows:')
print(df[['m1_contract', 'm12_contract', 'm1_price', 'm12_price', 'm1_ret', 'm12_ret']].tail())